# name: seba saud

# M2.Ex2: Automobile Fuel Efficiency

- Run: [**Open In Colab**](https://colab.research.google.com/github/HassanAlgoz/B5/blob/main/content/W3/M2/exercises/ex1_multi-reg.ipynb)

In [7]:
import pandas as pd
import sklearn

## Auto MPG Dataset

The Auto MPG Dataset is a classic benchmark for regression analysis in machine learning. It originally appeared in the 1983 American Statistical Association (ASA) Exposition and was later donated to the UCI Machine Learning Repository by Ross Quinlan in 1993.

The data consists of technical specifications for various car models from the late 1970s and early 1980s, primarily used to predict fuel efficiency (MPG).

- Features: `5` numerical, `3` categorical
- Target: `mpg` (miles per gallon)
- Size: `398` samples
- Source: [Auto MPG Dataset](https://archive.ics.uci.edu/dataset/9/auto+mpg)

### Step 1. Load the data

In [8]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/mpg.csv"
df = pd.read_csv(url)
df.rename(columns={'model_year': 'model year', 'name': 'car name'}, inplace=True)
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino


### Step 2.a Assign variables `X` to the features and `y` to the target

In [9]:
X = df.drop('mpg', axis=1)
y = df['mpg']

### Step 2.b print the type of each

In [10]:
print("Type of X:", type(X))
print("Type of y:", type(y))

Type of X: <class 'pandas.DataFrame'>
Type of y: <class 'pandas.Series'>


### Step 2.c identify whether the target is categorical or numerical & whether the task is regression or classification

mpg is numrical
and task is regression

### Step 3. Identify the number of samples and columns of both the data matrix and the target

In [ ]:
print("X shape (samples, features):", X.shape)
print("y shape (samples,):", y.shape)

### Step 4. Summarize the distribution of the data

- Use `describe()` for numerical features
- Use `describe()` for cateogrical features

In [ ]:
display(df.describe())
display(df.describe(include=['object', 'category']))

### Step 5. Plot each of the features vs the target

Hint use this: `sns.pairplot(adv,x_vars=['TV','Radio','Newspaper'],y_vars='Sales',height=6,aspect=0.7,kind='reg')`

In [ ]:

num_vars = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model year']
sns.pairplot(df, x_vars=num_vars, y_vars='mpg', height=4, aspect=0.8, kind='reg')
plt.show()

### Step 6. What is the relationship between the feature and the target? (increasing or decreasing or none)

1. `x=cylinders` and `y=mpg`
2. `x=displacement` and `y=mpg`
3. `x=horsepower` and `y=mpg`
4. `x=weight` and `y=mpg`
5. `x=acceleration` and `y=mpg`

1- decreasing
2- decreasing
3- decreasing               
4- decreasing
5- increasing

### Step 7. Define the pipeline with pre-processing steps

Raw data is rarely ready for use in ML models. We often need steps such as:

- Handling missing values
- Encoding categorical variables
- Scaling numerical variables

Use `ColumnTransformer` to separate the preprocessing steps of the numerical features from the categorical ones.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
num_features = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model year']
cat_features = ['origin', 'car name']
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')) 
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ])

predictor = LinearRegression()
]

)

In [ ]:
pipe = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", predictor),
    ]
)

### Step 8. Split the dataset into train and test sets

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Step 9.a Fit the pipeline on the training set

In [ ]:
pipe.fit(X_train, y_train)

### Step 9.b Identify the learned coefficients (for each feature) and the bias term

In [ ]:

coefficients = pipe.named_steps['regressor'].coef_
bias = pipe.named_steps['regressor'].intercept_

print("Number of Coefficients:", len(coefficients))
print("Bias (Intercept):", bias)


### Step 9.c how much `mpg` we gain if we decrease the weight of the automobile by `100kg`?

In [ ]:
scaler = pipe.named_steps['preprocessor'].transformers_[0][1].named_steps['scaler']
weight_scale = scaler.scale_[3] 
scaled_weight_coef = pipe.named_steps['regressor'].coef_[3]
unscaled_weight_coef = scaled_weight_coef / weight_scale
weight_change = -100 
mpg_gain = unscaled_weight_coef * weight_change
print(f"Unscaled Weight Coefficient: {unscaled_weight_coef:.5f}")
print(f"Gain in MPG if weight decreases by 100: {mpg_gain:.2f} MPG")

### Step 10. Evaluate the pipeline on the test set

In [ ]:
score = pipe.score(X_test, y_test)
print(f"Model R^2 Score on Test Set: {score:.4f}")